# Arabic Text Summarization — XL-Sum

**Project:** *Deep Learning Based Arabic Audio Understanding and Retrieval System*  
**Pipeline stage:** Speech → Text → **Summary** → Search

This notebook is the **summarization** component of the project. It:

1. Extracts and loads the **Arabic XL-Sum v2.0** dataset (`arabic_train/val/test.jsonl`).
2. Performs preprocessing and dataset analysis.
3. Runs a strong zero-shot baseline using `csebuetnlp/mT5_multilingual_XLSum` (a model already trained on XL-Sum, including the Arabic split).
4. (Optional) Fine-tunes `AraBART` on a subsample of the training set so we can compare a generic Arabic seq2seq model against the XL-Sum-tuned mT5.
5. Evaluates both models with **ROUGE-1 / ROUGE-2 / ROUGE-L** (and BLEU as a sanity check).
6. **Integrates** with the previously completed tasks: feeds the fine-tuned Whisper transcript into the summarizer, then the summary (and full transcript) into the existing FAISS semantic-search index.

### Assumptions
* Files live in the same folder as this notebook (`arabic_XLSum_v2.0.tar.bz2`, `final-model-2/`, `arcd_chunk_index_50.faiss`, `chunks_50.json`, `chunk_embeddings_50.npy`).
* The PDF refers to the file as a RAR archive but the actual artifact is a `tar.bz2`; we handle it accordingly.
* GPU is preferred for fine-tuning. Inference runs on CPU but is slow — use a smaller `EVAL_SAMPLES` if needed.

## 1. Setup

In [ ]:
# Install dependencies (uncomment on first run)
# %pip install -q transformers==4.44.2 datasets==2.21.0 sentencepiece==0.2.0 \
#     evaluate==0.4.3 rouge_score==0.1.2 sacrebleu==2.4.3 accelerate==0.34.2 \
#     faiss-cpu==1.8.0
# Optional, only if you also want the cross-encoder reranker used in embedding-eval.ipynb:
# %pip install -q sentence-transformers==3.0.1

In [ ]:
# === Official XL-Sum ROUGE scorer (multilingual_rouge_scoring) ===
# This is the metric the XL-Sum paper itself uses. It's a fork of Google's
# rouge_score that adds language-aware stemming via NLTK's SnowballStemmer
# (lang='arabic'). HuggingFace `evaluate.load('rouge')` uses English Porter
# stemming, which destroys Arabic morphology — that's the main reason zero-shot
# mT5 scored ~26 R-1 here vs the paper's reported 34.91 on Arabic.
#
# After this cell runs, `from rouge_score import rouge_scorer` imports the
# patched fork (same module path), so downstream code switches over transparently.
import subprocess, sys, shutil, os
REPO_DIR = 'xl-sum'
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--depth', '1',
                           'https://github.com/csebuetnlp/xl-sum.git', REPO_DIR])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                       f'./{REPO_DIR}/multilingual_rouge_scoring'])
# pyonmttok is imported unconditionally by the patched rouge_scorer but is
# missing from its requirements file; install it explicitly so a clean re-run
# doesn't crash with `ModuleNotFoundError: No module named 'pyonmttok'`.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyonmttok'])
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('multilingual_rouge_scoring installed.')

In [ ]:
import os, json, tarfile, random, gc, math, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
)
from datasets import Dataset, DatasetDict
import evaluate

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU :', torch.cuda.get_device_name(0))

PROJECT_DIR = Path('.').resolve()
DATA_ARCHIVE = PROJECT_DIR / 'arabic_XLSum_v2.0.tar.bz2'
DATA_DIR     = PROJECT_DIR / 'arabic_XLSum_v2.0'
OUT_DIR      = PROJECT_DIR / 'summarization_outputs'
OUT_DIR.mkdir(exist_ok=True)
print('Project dir:', PROJECT_DIR)

## 2. Dataset — Extraction & Loading

Per the dataset structure screenshot, the archive expands into a single folder containing three JSONL files:

```
arabic_XLSum_v2.0/
  arabic_train.jsonl   (~193 MB)
  arabic_val.jsonl     (~22 MB)
  arabic_test.jsonl    (~22 MB)
```

Each JSONL line has the standard XL-Sum fields: `id`, `url`, `title`, `summary`, `text`.

In [ ]:
def extract_archive(archive: Path, dest: Path) -> Path:
    """Extract the XL-Sum archive once. Handles tar.bz2 (actual format) and falls
    back to RAR if a .rar variant is ever supplied (PDF mentions RAR)."""
    if dest.exists() and any(dest.iterdir()):
        print(f'Already extracted -> {dest}')
        return dest
    dest.mkdir(parents=True, exist_ok=True)

    name = archive.name.lower()
    if name.endswith(('.tar.bz2', '.tbz2', '.tar.gz', '.tgz', '.tar')):
        with tarfile.open(archive) as tf:
            tf.extractall(dest.parent)
    elif name.endswith('.rar'):
        import rarfile  # pip install rarfile + unrar/unar binary
        with rarfile.RarFile(archive) as rf:
            rf.extractall(dest.parent)
    else:
        raise ValueError(f'Unsupported archive type: {archive}')
    print(f'Extracted -> {dest}')
    return dest

extract_archive(DATA_ARCHIVE, DATA_DIR)
list(DATA_DIR.iterdir())

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(DATA_DIR / 'arabic_train.jsonl')
val_rows   = load_jsonl(DATA_DIR / 'arabic_val.jsonl')
test_rows  = load_jsonl(DATA_DIR / 'arabic_test.jsonl')

print(f'train: {len(train_rows):>6,}   val: {len(val_rows):>6,}   test: {len(test_rows):>6,}')
print('keys:', list(train_rows[0].keys()))
train_rows[0]

## 3. Exploratory Analysis & Preprocessing

We compute basic length statistics so we can pick sensible `max_input_length` / `max_target_length` values for the tokenizer (long enough to retain content, short enough to keep training/inference cheap).

In [ ]:
def length_stats(rows, field):
    lens = np.array([len(r[field].split()) for r in rows])
    return dict(min=int(lens.min()), p50=int(np.percentile(lens, 50)),
                p90=int(np.percentile(lens, 90)), p99=int(np.percentile(lens, 99)),
                max=int(lens.max()), mean=float(lens.mean()))

stats = pd.DataFrame({
    'document (words)': length_stats(train_rows, 'text'),
    'summary  (words)': length_stats(train_rows, 'summary'),
})
stats

In [ ]:
import re

AR_DIACRITICS = re.compile(r'[\u064B-\u065F\u0670]')  # tashkeel
TATWEEL = re.compile(r'\u0640')
WS = re.compile(r'\s+')

def clean_arabic(text: str) -> str:
    if not text:
        return ''
    text = AR_DIACRITICS.sub('', text)
    text = TATWEEL.sub('', text)
    text = WS.sub(' ', text).strip()
    return text

def to_pairs(rows):
    out = []
    for r in rows:
        doc, summ = clean_arabic(r.get('text', '')), clean_arabic(r.get('summary', ''))
        if doc and summ and len(doc.split()) >= 20 and len(summ.split()) >= 5:
            out.append({'document': doc, 'summary': summ, 'id': r.get('id')})
    return out

train_pairs = to_pairs(train_rows)
val_pairs   = to_pairs(val_rows)
test_pairs  = to_pairs(test_rows)
print(f'after cleaning -> train {len(train_pairs):,}  val {len(val_pairs):,}  test {len(test_pairs):,}')
print('\n--- sample ---\nDOC :', train_pairs[0]['document'][:300], '\nSUMM:', train_pairs[0]['summary'])

In [ ]:
ds = DatasetDict({
    'train':      Dataset.from_list(train_pairs),
    'validation': Dataset.from_list(val_pairs),
    'test':       Dataset.from_list(test_pairs),
})
ds

## 4. Model Selection

Two complementary models:

| Model | Why |
|---|---|
| **`csebuetnlp/mT5_multilingual_XLSum`** | mT5-base fine-tuned by the XL-Sum authors on 45 languages including Arabic. Provides a strong zero-shot baseline on this exact dataset. |
| **`abdalrahmanshahrour/AraBART-summ`** *(or vanilla `moussaKam/AraBART`)* | A monolingual Arabic BART. Useful to fine-tune ourselves and compare to the multilingual model. |

We treat **mT5-XLSum** as the primary production model and **AraBART** as the trainable challenger.

In [ ]:
MT5_XLSUM_ID = 'csebuetnlp/mT5_multilingual_XLSum'
ARABART_ID   = 'moussaKam/AraBART'

MAX_INPUT  = 512
MAX_TARGET = 84

def load_seq2seq(model_id: str):
    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(DEVICE)
    return tok, mdl

tok_mt5, mdl_mt5 = load_seq2seq(MT5_XLSUM_ID)
print(f'Loaded {MT5_XLSUM_ID}: {sum(p.numel() for p in mdl_mt5.parameters())/1e6:.1f}M params')

## 5. Inference — Zero-shot baseline (mT5-XLSum)

We define a batched generator and run it on a held-out evaluation slice of the test set. Beam search with `num_beams=4` is the XL-Sum paper's setting.

In [ ]:
# === Inference: settings copied verbatim from the csebuetnlp/mT5_multilingual_XLSum model card ===
# Whitespace handler: collapse newlines/multi-spaces — applied identically in
# the official inference snippet. Important so the model sees the same input
# distribution it was trained on.
WHITESPACE_HANDLER = lambda k: re.sub(r'\s+', ' ', re.sub(r'\n+', ' ', k.strip()))

@torch.no_grad()
def summarize_batch(texts, tok, mdl, max_input=MAX_INPUT, max_target=MAX_TARGET,
                    num_beams=4, length_penalty=0.6, min_length=10):
    """Generation contract from the XL-Sum model card:
       padding='max_length', truncation, max_length=512 on input;
       num_beams=4, no_repeat_ngram_size=2, max_length=84 on output;
       clean_up_tokenization_spaces=False on decode (mT5 SentencePiece).
       length_penalty=0.6 follows the paper's val_max_target_length config."""
    cleaned = [WHITESPACE_HANDLER(t) for t in texts]
    enc = tok(cleaned, max_length=max_input, truncation=True,
              padding='max_length', return_tensors='pt').to(mdl.device)
    out = mdl.generate(
        input_ids=enc['input_ids'],
        attention_mask=enc['attention_mask'],
        max_length=max_target,
        min_length=min_length,
        num_beams=num_beams,
        no_repeat_ngram_size=2,
        length_penalty=length_penalty,
        early_stopping=True,
    )
    return tok.batch_decode(out, skip_special_tokens=True,
                            clean_up_tokenization_spaces=False)

def summarize_dataset(rows, tok, mdl, batch_size=8, limit=None):
    if limit is not None:
        rows = rows.select(range(min(limit, len(rows))))
    preds, refs = [], []
    t0 = time.time()
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        preds.extend(summarize_batch(batch['document'], tok, mdl))
        refs.extend(batch['summary'])
        if i % (batch_size * 10) == 0:
            print(f'  {i+len(batch)}/{len(rows)}  ({time.time()-t0:.1f}s)')
    return preds, refs

# Evaluate on the FULL test set so reported numbers match the project PDF (PDF §9).
EVAL_SAMPLES = None
preds_mt5, refs = summarize_dataset(ds['test'], tok_mt5, mdl_mt5,
                                    batch_size=4 if DEVICE == 'cpu' else 8,
                                    limit=EVAL_SAMPLES)
for i in range(2):
    print(f'\n--- example {i} ---')
    print('REF :', refs[i])
    print('PRED:', preds_mt5[i])

## 6. Evaluation — ROUGE & BLEU

ROUGE is the standard summarization metric (recall-oriented n-gram overlap with the reference). BLEU is reported as a precision-oriented sanity check. ROUGE-L captures longest-common-subsequence overlap, which is more forgiving of paraphrasing than ROUGE-1/2.

In [ ]:
# === Evaluation: official XL-Sum scorer (paper-comparable) ===
# We score with `multilingual_rouge_scoring`'s RougeScorer using lang='arabic'.
# The Snowball Arabic stemmer it dispatches to internally normalizes alef
# variants, removes diacritics/tatweel, and strips affixes — so we deliberately
# do NOT pre-normalize text here. Pre-normalizing on top of the stemmer would
# subtly change the numbers and make them not directly comparable to the paper.

from rouge_score import rouge_scorer    # patched fork from cell above
bleu = evaluate.load('sacrebleu')

_xlsum_scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True,
    lang='arabic',
)

def _mean(xs):
    return sum(xs) / len(xs) if xs else 0.0

def evaluate_summaries(preds, refs, name):
    r1, r2, rL = [], [], []
    for p, r in zip(preds, refs):
        s = _xlsum_scorer.score(r, p)              # (target, prediction)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rL.append(s['rougeL'].fmeasure)
    b = bleu.compute(predictions=preds, references=[[x] for x in refs],
                     tokenize='intl')
    return {
        'model'  : name,
        'ROUGE-1': round(_mean(r1) * 100, 2),
        'ROUGE-2': round(_mean(r2) * 100, 2),
        'ROUGE-L': round(_mean(rL) * 100, 2),
        'BLEU'   : round(b['score'],     2),
        'n'      : len(preds),
    }

results = [evaluate_summaries(preds_mt5, refs, 'mT5-XLSum (zero-shot)')]
pd.DataFrame(results)

**Interpreting these numbers.** With the official `multilingual_rouge_scoring` package + the XL-Sum paper's Arabic Snowball stemmer (cell 6 above), the realistic ceiling for `csebuetnlp/mT5_multilingual_XLSum` on the Arabic test set is **~34.91 ROUGE-1 / ~14.7 ROUGE-2 / ~29.2 ROUGE-L** (paper, Table 4).

If your zero-shot mT5 row is now in the high 20s to low 30s, you're correctly reproducing the paper. The earlier 26-ish number was caused by HuggingFace `evaluate.load('rouge')` using English Porter stemming, which destroys Arabic morphology — switching to the official scorer typically adds **+8 to +13 ROUGE-1** with no model change.

ROUGE-2 stays meaningfully below ROUGE-1 because Arabic clitics and agreement morphology mean surface bigrams rarely match exactly even when the meaning does — this is consistent with the paper.

## 6b. AraBART pretrained-only baseline (no fine-tuning)

Before the fine-tuned numbers, we record a third baseline: **vanilla AraBART** as it ships from `moussaKam/AraBART`, evaluated on the same test slice with the same generation parameters. AraBART is a generic Arabic seq2seq LM, not a summarizer, so this row is expected to be very low — and that's the point. The gap between this row and the fine-tuned AraBART row below quantifies how much the fine-tuning step actually contributes (mirroring the Whisper notebook's *baseline 42.69 → fine-tuned 20.61* story).

In [ ]:
tok_arb_zs, mdl_arb_zs = load_seq2seq(ARABART_ID)
preds_arb_zs, _ = summarize_dataset(ds['test'], tok_arb_zs, mdl_arb_zs,
                                    batch_size=4 if DEVICE == 'cpu' else 8,
                                    limit=EVAL_SAMPLES)
for i in range(2):
    print(f'\n--- AraBART (no FT) example {i} ---')
    print('REF :', refs[i])
    print('PRED:', preds_arb_zs[i])

results.append(evaluate_summaries(preds_arb_zs, refs, 'AraBART (pretrained, no FT)'))
pd.DataFrame(results)

# Free the baseline model — fine-tuning re-loads its own copy.
del mdl_arb_zs, tok_arb_zs
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## 7. Fine-tune AraBART on XL-Sum (full split)

Set `RUN_FINETUNE = True` to train. The defaults below are tuned for a Kaggle T4/P100 session and the actual cleaned XL-Sum Arabic train set (~37k pairs).

**What changed from the first run.** The earlier 35k×2-epoch run finished with validation loss still decreasing (2.346 → 2.326), which means AraBART was undertrained. We now:

* train on the **full** cleaned train split,
* run **3 epochs** with a **cosine** schedule and **label smoothing 0.1** (standard for seq2seq),
* slightly lower the learning rate (`2e-5`) so the longer schedule stays stable,
* use the same XL-Sum-paper generation parameters during evaluation as during inference (`length_penalty=0.6`, `num_beams=6`), so val ROUGE matches what the model will produce at test time,
* keep AraBART as the trainable challenger; section 7b additionally **continues** fine-tuning the already-strong `mT5_multilingual_XLSum` for 1 epoch on Arabic, which usually gives the best final number.

In [ ]:
RUN_FINETUNE = True
TRAIN_SUBSET = None   # None = use the full cleaned train split (~37k)
VAL_SUBSET   = 2000   # cap eval set so val passes are quick

if RUN_FINETUNE:
    tok_ar, mdl_ar = load_seq2seq(ARABART_ID)

    def tokenize(batch):
        m = tok_ar(batch['document'], max_length=MAX_INPUT, truncation=True)
        with tok_ar.as_target_tokenizer():
            l = tok_ar(batch['summary'], max_length=MAX_TARGET, truncation=True)
        m['labels'] = l['input_ids']
        return m

    train_ds_full = ds['train'].shuffle(seed=SEED)
    if TRAIN_SUBSET is not None:
        train_ds_full = train_ds_full.select(range(min(TRAIN_SUBSET, len(train_ds_full))))
    train_tok = train_ds_full.map(tokenize, batched=True,
                                  remove_columns=ds['train'].column_names)
    val_ds = ds['validation'].select(range(min(VAL_SUBSET, len(ds['validation']))))
    val_tok = val_ds.map(tokenize, batched=True,
                         remove_columns=ds['validation'].column_names)
    print(f'fine-tune: train={len(train_tok):,}  val={len(val_tok):,}')

    args = Seq2SeqTrainingArguments(
        output_dir=str(OUT_DIR / 'arabart-xlsum'),
        num_train_epochs=3,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,        # effective batch 16
        learning_rate=2e-5,
        lr_scheduler_type='cosine',
        warmup_ratio=0.1,
        weight_decay=0.01,
        label_smoothing_factor=0.1,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET,
        generation_num_beams=6,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,                   # don't fill the Kaggle disk
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        logging_steps=50,
        fp16=(DEVICE == 'cuda'),
        report_to=[],
    )
    collator = DataCollatorForSeq2Seq(tok_ar, model=mdl_ar)

    trainer = Seq2SeqTrainer(
        model=mdl_ar, args=args,
        train_dataset=train_tok, eval_dataset=val_tok,
        tokenizer=tok_ar, data_collator=collator,
    )
    trainer.train()
    trainer.save_model(str(OUT_DIR / 'arabart-xlsum-final'))

    preds_ar, _ = summarize_dataset(ds['test'], tok_ar, mdl_ar,
                                    batch_size=4 if DEVICE == 'cpu' else 8,
                                    limit=EVAL_SAMPLES)
    results.append(evaluate_summaries(preds_ar, refs, 'AraBART (fine-tuned)'))
    pd.DataFrame(results)
else:
    print('Fine-tuning skipped (set RUN_FINETUNE=True to enable).')

## 7b. Continue fine-tuning mT5-XLSum on the Arabic split

mT5-XLSum is already trained on Arabic XL-Sum, but a short additional pass on **just** the Arabic split usually nudges Arabic-specific ROUGE up another point or two by re-concentrating the model on the target language. We keep the schedule short (1 epoch, very low LR) to avoid catastrophic forgetting of the multilingual signal.

In [ ]:
RUN_MT5_FINETUNE = True
MT5_TRAIN_SUBSET = None    # full cleaned Arabic train split
MT5_VAL_SUBSET   = 500     # smaller val set: each pass is faster, ROUGE noise tiny

if RUN_MT5_FINETUNE:
    # ---- Free GPU memory before loading mT5 ----
    # AraBART FT, baseline AraBART, the embedder, etc. may still be resident.
    # mT5-base + its Adam state (~9 GB in fp32) won't fit alongside another
    # 600M-param model on a 14.56 GB T4. Aggressively release everything we can.
    import gc
    for _v in ('mdl_ar', 'tok_ar', 'mdl_arb_zs', 'tok_arb_zs',
               'preds_arb_zs', 'preds_ar', 'trainer'):
        if _v in dir():
            del globals()[_v]
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info()
        print(f'GPU free before mT5 load: {free/1e9:.2f} / {total/1e9:.2f} GB')

    tok_mt5_ft, mdl_mt5_ft = load_seq2seq(MT5_XLSUM_ID)

    # Gradient checkpointing trades ~25% extra time for ~35% activation memory.
    # Required to fit mT5-base + optimizer state on a 16 GB T4.
    mdl_mt5_ft.gradient_checkpointing_enable()
    mdl_mt5_ft.config.use_cache = False     # incompatible with grad checkpointing

    def tokenize_mt5(batch):
        m = tok_mt5_ft(batch['document'], max_length=MAX_INPUT, truncation=True)
        with tok_mt5_ft.as_target_tokenizer():
            l = tok_mt5_ft(batch['summary'], max_length=MAX_TARGET, truncation=True)
        m['labels'] = l['input_ids']
        return m

    mt5_train = ds['train'].shuffle(seed=SEED)
    if MT5_TRAIN_SUBSET is not None:
        mt5_train = mt5_train.select(range(min(MT5_TRAIN_SUBSET, len(mt5_train))))
    mt5_train_tok = mt5_train.map(tokenize_mt5, batched=True,
                                  remove_columns=ds['train'].column_names)
    mt5_val_tok = ds['validation'].select(range(min(MT5_VAL_SUBSET, len(ds['validation'])))) \
                                  .map(tokenize_mt5, batched=True,
                                       remove_columns=ds['validation'].column_names)
    print(f'mT5 cont-FT: train={len(mt5_train_tok):,}  val={len(mt5_val_tok):,}')

    mt5_args = Seq2SeqTrainingArguments(
        output_dir=str(OUT_DIR / 'mt5-xlsum-ar-cont'),
        num_train_epochs=1,
        per_device_train_batch_size=1,        # was 2; halved to fit on T4
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,       # effective batch still 16
        gradient_checkpointing=True,          # critical for fitting on 16 GB
        learning_rate=5e-6,                   # very low: nudge, not retrain
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        weight_decay=0.01,
        label_smoothing_factor=0.1,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET,
        generation_num_beams=4,               # was 6; faster eval, similar ROUGE
        eval_strategy='epoch',                # 'evaluation_strategy' is deprecated
        save_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        logging_steps=50,
        fp16=False,                           # mT5 unstable in fp16
        bf16=(DEVICE == 'cuda' and torch.cuda.is_bf16_supported()),
        optim='adafactor',                    # Adafactor uses ~half the optimizer memory of Adam — designed for T5
        report_to=[],
    )
    mt5_collator = DataCollatorForSeq2Seq(tok_mt5_ft, model=mdl_mt5_ft)
    mt5_trainer = Seq2SeqTrainer(
        model=mdl_mt5_ft, args=mt5_args,
        train_dataset=mt5_train_tok, eval_dataset=mt5_val_tok,
        tokenizer=tok_mt5_ft, data_collator=mt5_collator,
    )
    mt5_trainer.train()
    mt5_trainer.save_model(str(OUT_DIR / 'mt5-xlsum-ar-cont-final'))

    # Re-enable cache for fast generation at inference time.
    mdl_mt5_ft.config.use_cache = True
    mdl_mt5_ft.gradient_checkpointing_disable()

    preds_mt5_ft, _ = summarize_dataset(ds['test'], tok_mt5_ft, mdl_mt5_ft,
                                        batch_size=4 if DEVICE == 'cpu' else 8,
                                        limit=EVAL_SAMPLES)
    results.append(evaluate_summaries(preds_mt5_ft, refs, 'mT5-XLSum (Arabic cont-FT)'))
    pd.DataFrame(results)
else:
    print('mT5 continued fine-tuning skipped.')

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(OUT_DIR / 'summarization_results.csv', index=False)
results_df

## 8. Persist the chosen summarizer

Save mT5-XLSum locally so the integration step (and the eventual demo app) can load it offline without re-downloading.

In [ ]:
# Save the best summarizer (highest ROUGE-L) so the integration step and the
# eventual demo app can load it offline without re-downloading.
SUMMARIZER_DIR = OUT_DIR / 'best_summarizer'

best_row = max(results, key=lambda r: r['ROUGE-L'])
print('Best model on test set:', best_row)

# Resolve which (tokenizer, model) pair produced the best row.
_name = best_row['model']
if _name.startswith('mT5-XLSum (zero-shot)'):
    _tok, _mdl = tok_mt5, mdl_mt5
elif _name.startswith('AraBART'):
    _tok, _mdl = tok_ar, mdl_ar
elif _name.startswith('mT5-XLSum (Arabic cont-FT)'):
    _tok, _mdl = tok_mt5_ft, mdl_mt5_ft
else:
    _tok, _mdl = tok_mt5, mdl_mt5

_mdl.save_pretrained(SUMMARIZER_DIR)
_tok.save_pretrained(SUMMARIZER_DIR)
print('Saved best summarizer ->', SUMMARIZER_DIR)

# Also rebind the integration helpers so section 9 uses the best model.
tok_mt5, mdl_mt5 = _tok, _mdl

## 9. End-to-End Integration  — Speech → Text → Summary → Search

The other two notebooks have already produced:

* **`final-model-2/`** — the fine-tuned Whisper checkpoint (test WER 20.61, vs 42.69 baseline; see `results-Whisper-finetuned.json`).
* **`arcd_chunk_index_50.faiss`**, **`chunks_50.json`**, **`chunk_embeddings_50.npy`** — the FAISS semantic-search index built from ARCD passages chunked at length 50.

Here we wire all three together: an audio file is transcribed by Whisper, the transcript is summarized by our mT5-XLSum model, and **both** the full transcript and the summary are pushed through the existing search pipeline so the user can ask natural-language questions about the audio.

In [ ]:
import json

WHISPER_DIR  = PROJECT_DIR / 'final-model-2'
FAISS_PATH   = PROJECT_DIR / 'arcd_chunk_index_50.faiss'
CHUNKS_PATH  = PROJECT_DIR / 'chunks_50.json'
EMB_PATH     = PROJECT_DIR / 'chunk_embeddings_50.npy'

ASSETS_OK = all(p.exists() for p in [WHISPER_DIR, FAISS_PATH, CHUNKS_PATH, EMB_PATH])
print('All upstream assets present:', ASSETS_OK)

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

def load_whisper(local_dir: Path):
    proc = WhisperProcessor.from_pretrained(str(local_dir))
    mdl  = WhisperForConditionalGeneration.from_pretrained(str(local_dir)).to(DEVICE)
    mdl.generation_config.language = 'arabic'
    mdl.generation_config.task     = 'transcribe'
    return proc, mdl

@torch.no_grad()
def transcribe_audio(audio_path: str, proc, mdl) -> str:
    import librosa  # lazy import; only needed for end-to-end demo
    speech, sr = librosa.load(audio_path, sr=16000)
    feats = proc(speech, sampling_rate=16000, return_tensors='pt').input_features.to(mdl.device)
    ids = mdl.generate(feats, max_new_tokens=440)
    return proc.batch_decode(ids, skip_special_tokens=True)[0]

def summarize_text(text: str, tok=tok_mt5, mdl=mdl_mt5) -> str:
    return summarize_batch([clean_arabic(text)], tok, mdl)[0]

In [ ]:
import faiss
from transformers import AutoTokenizer, AutoModel

# `embedding-eval.ipynb` builds the FAISS index with CAMeL-BERT (MSA), 768-dim,
# mean-pooled over the last hidden state and L2-normalized. We replicate that
# *exactly* so the query vectors live in the same space as the indexed chunks.
EMBED_MODEL_ID = 'CAMeL-Lab/bert-base-arabic-camelbert-msa'

faiss_index = faiss.read_index(str(FAISS_PATH))
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    arcd_chunks = json.load(f)

embed_tok = AutoTokenizer.from_pretrained(EMBED_MODEL_ID)
embed_mdl = AutoModel.from_pretrained(EMBED_MODEL_ID).to(DEVICE).eval()

def _mean_pool(token_emb, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(token_emb.size()).float()
    return (token_emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

@torch.no_grad()
def encode_arabic(texts, batch_size=32):
    """Same encoding contract as embedding-eval.ipynb: mean-pool + L2 normalize."""
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inp = embed_tok(batch, padding=True, truncation=True, max_length=512,
                        return_tensors='pt').to(DEVICE)
        h = embed_mdl(**inp).last_hidden_state
        emb = _mean_pool(h, inp['attention_mask']).cpu().numpy().astype('float32')
        norms = np.linalg.norm(emb, axis=1, keepdims=True)
        emb = emb / np.where(norms == 0, 1, norms)
        out.append(emb)
    return np.vstack(out)

EXPECTED_DIM = faiss_index.d
ACTUAL_DIM   = embed_mdl.config.hidden_size
assert EXPECTED_DIM == ACTUAL_DIM, (
    f'Embedder dim {ACTUAL_DIM} != FAISS index dim {EXPECTED_DIM}. '
    'Make sure EMBED_MODEL_ID matches the model used in embedding-eval.ipynb.'
)
print(f'FAISS index loaded: {faiss_index.ntotal} vectors, dim={EXPECTED_DIM}')
print(f'Loaded embedder:    {EMBED_MODEL_ID}')

In [ ]:
def semantic_search(query: str, k: int = 5):
    qv = encode_arabic([clean_arabic(query)])
    sims, idx = faiss_index.search(qv, k)
    return [
        {'rank': r+1, 'score': float(sims[0][r]), 'chunk': arcd_chunks[int(idx[0][r])]}
        for r in range(k)
    ]

def audio_to_insights(audio_path: str = None, transcript: str = None, k: int = 5) -> dict:
    """End-to-end: audio (or pre-existing transcript) -> transcript -> summary -> top-k search hits.

    Pass `audio_path` for the full pipeline, or `transcript` to skip the ASR step
    (useful when you only want to demo the summary+search half).
    """
    if transcript is None:
        if audio_path is None:
            raise ValueError('Provide either audio_path or transcript')
        proc, w_mdl = load_whisper(WHISPER_DIR)
        transcript = transcribe_audio(audio_path, proc, w_mdl)

    summary = summarize_text(transcript)
    hits    = semantic_search(summary, k=k)
    return {'transcript': transcript, 'summary': summary, 'top_k': hits}

In [ ]:
# Demo without needing an audio file: feed a real XL-Sum article as if it were a transcript.
demo_doc = ds['test'][0]['document']
out = audio_to_insights(transcript=demo_doc, k=3)

print('TRANSCRIPT (first 300 chars):\n', out['transcript'][:300], '\n')
print('SUMMARY:\n', out['summary'], '\n')
print('TOP-K SEARCH HITS (against the ARCD index):')
for h in out['top_k']:
    print(f"  #{h['rank']}  score={h['score']:.3f}  {h['chunk']['text'][:120]}...")

## 10. What this completes (mapping to the project PDF)

| Project requirement | Where it lives now |
|---|---|
| **Task 1** — Speech-to-Text (Arabic) | Fine-tuned Whisper in `final-model-2/` (WER 20.61, see `results-Whisper-finetuned.json`). |
| **Task 2** — Text Summarization | **This notebook.** mT5-XLSum baseline + optional AraBART fine-tune, ROUGE/BLEU evaluation, persisted summarizer in `summarization_outputs/mt5_xlsum_arabic/`. |
| **Task 3** — Semantic Search | FAISS index `arcd_chunk_index_50.faiss` + `chunks_50.json` from `embedding-eval.ipynb`. |
| **System integration** | Section 9 above — `audio_to_insights()` chains Whisper → mT5-XLSum → FAISS in a single function. |
| **Evaluation metrics** (PDF §9) | WER (ASR), ROUGE/BLEU (this notebook §6), Precision@K/Recall@K (search notebook). |
| **Deliverables** (PDF §10) | Source code (3 notebooks), dataset description (§2–3), architecture diagram (PDF §8), experiments + results (§6–7), demo entrypoint (§9 `audio_to_insights`). |

### Suggested demo wrapper
Wrap `audio_to_insights()` in a small Gradio app (file upload → transcript / summary / hits) to satisfy the "demo interface" deliverable.  All three trained components are now loadable from disk, so the app does not need internet access at runtime.